# Before you start

Replace the Database and Schema on lines 12 & 13 of cell 1 with a DB and Schema you have write privilges for or create the ones I have and the Demo should run top down

In [ ]:
from snowflake.snowpark import Session
from snowflake.snowpark.context import get_active_session

try:
    # Try to get an existing session (Snowflake Notebooks)
    session = get_active_session()
except Exception:
    # Use a specific connection defined in your config.toml
    # Replace "chase_demo" with the header name from your TOML file
    session = Session.builder.config("connection_name", "chase_demo").create()

session.use_database("FSI_DEMO_DB")
session.use_schema("QRI")

In [ ]:
create or replace stage RESEARCH_NOTES 
	DIRECTORY = ( ENABLE = true 
                  AUTO_REFRESH = TRUE) 
	ENCRYPTION = ( TYPE = 'SNOWFLAKE_SSE' );

In [ ]:
MY_STAGE = 'RESEARCH_NOTES'
MY_FILE_NAME = "research_notes/*.pdf"


# Upload the file to a stage.
put_result = session.file.put(MY_FILE_NAME, MY_STAGE, auto_compress=False,overwrite=True)


In [ ]:
alter stage RESEARCH_NOTES refresh;

In [ ]:
CREATE OR REPLACE TABLE ANALYST_RESEARCH_NOTES (
    NOTE_ID             INT AUTOINCREMENT,
    FILE_PATH           VARCHAR(500),
    PUBLISH_DATE        DATE,
    ANALYST_NAME        VARCHAR(100),
    TICKER              VARCHAR(10),
    COMPANY_NAME        VARCHAR(200),
    SECTOR              VARCHAR(100),
    NOTE_TYPE           VARCHAR(50),
    TITLE               VARCHAR(500),
    BODY                TEXT,
    RATING              VARCHAR(20),
    RATING_CHANGE       VARCHAR(30),
    CONVICTION          VARCHAR(10),
    PRICE_TARGET        VARCHAR(20),
    RAW_EXTRACT         VARIANT,
    EXTRACTED_AT        TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

In [ ]:
SELECT
    relative_path                                  AS file_path,
    AI_PARSE_DOCUMENT(
        TO_FILE('@RESEARCH_NOTES', relative_path),
        {'mode': 'LAYOUT'}
    ):content::STRING                              AS parsed_body
FROM DIRECTORY(@RESEARCH_NOTES)
WHERE relative_path LIKE '%.pdf'
LIMIT 1;

In [ ]:
SELECT
    relative_path                                  AS file_path,
    AI_EXTRACT(
        file => TO_FILE('@RESEARCH_NOTES', relative_path),
        responseFormat => {
            'publish_date':   'What is the date of this research note? Return in YYYY-MM-DD format.',
            'analyst_name':   'Who is the analyst that authored this research note?',
            'ticker':         'What is the stock ticker symbol? If this covers multiple stocks, return MULTI.',
            'company_name':   'What is the company name? If multiple companies, return Cross-Sector.',
            'sector':         'What sector or industry does this note cover?',
            'note_type':      'What type of research note is this? Return one of: Earnings Review, Initiation, Sector Update, Quant Signal Change, Risk Alert.',
            'title':          'What is the title or headline of this research note?',
            'rating':         'What is the analyst rating? Return one of: Overweight, Neutral, Underweight, N/A.',
            'rating_change':  'What is the rating change action? Return one of: Upgrade, Downgrade, Maintain, Initiation, N/A.',
            'conviction':     'What is the conviction level? Return one of: High, Medium, Low, N/A.',
            'price_target':   'What is the price target? Return the dollar amount or N/A if not specified.'
        }
    ) AS extracted
FROM DIRECTORY(@RESEARCH_NOTES)
WHERE relative_path LIKE '%.pdf'
LIMIT 1;

In [ ]:
INSERT INTO ANALYST_RESEARCH_NOTES (
    FILE_PATH,
    PUBLISH_DATE,
    ANALYST_NAME,
    TICKER,
    COMPANY_NAME,
    SECTOR,
    NOTE_TYPE,
    TITLE,
    BODY,
    RATING,
    RATING_CHANGE,
    CONVICTION,
    PRICE_TARGET,
    RAW_EXTRACT
)
SELECT
    d.relative_path                                                         AS FILE_PATH,
    TRY_TO_DATE(e.extracted:response:publish_date::STRING)                  AS PUBLISH_DATE,
    e.extracted:response:analyst_name::STRING                                AS ANALYST_NAME,
    e.extracted:response:ticker::STRING                                      AS TICKER,
    e.extracted:response:company_name::STRING                                AS COMPANY_NAME,
    e.extracted:response:sector::STRING                                      AS SECTOR,
    e.extracted:response:note_type::STRING                                   AS NOTE_TYPE,
    e.extracted:response:title::STRING                                       AS TITLE,
    p.parsed:content::STRING                                                 AS BODY,
    e.extracted:response:rating::STRING                                      AS RATING,
    e.extracted:response:rating_change::STRING                               AS RATING_CHANGE,
    e.extracted:response:conviction::STRING                                  AS CONVICTION,
    e.extracted:response:price_target::STRING                                AS PRICE_TARGET,
    e.extracted                                                              AS RAW_EXTRACT
FROM DIRECTORY(@RESEARCH_NOTES) d,
LATERAL (
    SELECT AI_EXTRACT(
        file => TO_FILE('@RESEARCH_NOTES', d.relative_path),
        responseFormat => {
            'publish_date':   'What is the date of this research note? Return in YYYY-MM-DD format.',
            'analyst_name':   'Who is the analyst that authored this research note?',
            'ticker':         'What is the stock ticker symbol? If this covers multiple stocks, return MULTI.',
            'company_name':   'What is the company name? If multiple companies, return Cross-Sector.',
            'sector':         'What sector or industry does this note cover?',
            'note_type':      'What type of research note is this? Return one of: Earnings Review, Initiation, Sector Update, Quant Signal Change, Risk Alert.',
            'title':          'What is the title or headline of this research note?',
            'rating':         'What is the analyst rating? Return one of: Overweight, Neutral, Underweight, N/A.',
            'rating_change':  'What is the rating change action? Return one of: Upgrade, Downgrade, Maintain, Initiation, N/A.',
            'conviction':     'What is the conviction level? Return one of: High, Medium, Low, N/A.',
            'price_target':   'What is the price target? Return the dollar amount or N/A if not specified.'
        }
    ) AS extracted
) e,
LATERAL (
    SELECT AI_PARSE_DOCUMENT(
        TO_FILE('@RESEARCH_NOTES', d.relative_path),
        {'mode': 'LAYOUT'}
    ) AS parsed
) p
WHERE d.relative_path LIKE '%.pdf';

In [ ]:
SELECT
    NOTE_ID,
    FILE_PATH,
    PUBLISH_DATE,
    ANALYST_NAME,
    TICKER,
    COMPANY_NAME,
    SECTOR,
    NOTE_TYPE,
    TITLE,
    RATING,
    RATING_CHANGE,
    CONVICTION,
    PRICE_TARGET,
    LEFT(BODY, 200)    AS BODY_PREVIEW,
    EXTRACTED_AT
FROM ANALYST_RESEARCH_NOTES
ORDER BY PUBLISH_DATE DESC;

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE RESEARCH_NOTES_SEARCH
  ON BODY
  ATTRIBUTES TICKER, COMPANY_NAME, SECTOR, NOTE_TYPE, RATING, ANALYST_NAME
  WAREHOUSE = 'DEMO_WH'
  TARGET_LAG = '1 hour'
  AS (
    SELECT
        NOTE_ID,
        PUBLISH_DATE,
        ANALYST_NAME,
        TICKER,
        COMPANY_NAME,
        SECTOR,
        NOTE_TYPE,
        TITLE,
        BODY,
        RATING,
        RATING_CHANGE,
        CONVICTION
    FROM ANALYST_RESEARCH_NOTES
  );

# Use UI for semantic views & Agent

### Semantic View 1: `PORTFOLIO_ANALYTICS`

Covers `PORTFOLIO_HOLDINGS`, `FUND_PERFORMANCE`, and `TRADE_HISTORY`.

### Semantic View 2: `QUANT_SIGNAL_ANALYTICS`

Covers `QUANT_SIGNALS`.

### Agent tools
- Add both semantic views
- Add search service 
Search Description - description as "Search internal analyst research notes, earnings reviews, sector updates, quant signal change commentary, and risk alerts. Notes include analyst ratings (Overweight/Neutral/Underweight), conviction levels, and detailed qualitative analysis."
- Orchestraion Instructions: "You are a quantitative research assistant for the QRI (Quantitative Research and Investing) division. You help portfolio managers and analysts explore portfolio holdings, fund performance, quantitative factor signals, and internal research notes. When asked about specific stocks, always check both the structured data (signals, holdings) and unstructured research notes to provide a complete picture. When discussing performance, always include benchmark comparisons. Cite specific analyst notes when relevant."
- Response - "Format responses clearly with tables for numerical data. When presenting rankings or scores, include the date context. Always specify which fund you are referencing. If a question spans both funds, present side-by-side comparisons."



### Portfolio Overview & Quant Signals (Structured Data)

1. "What are the top 10 holdings by weight in the Advanced US Equity Fund?"
2. "How has the Absolute Income Fund performed YTD vs the Bloomberg Agg?"
3. "Which stocks received composite score upgrades this week?"

### Research Intelligence (Unstructured Search)

4. "What did our analysts say about NVIDIA after Q4 earnings?"
5. "Find any risk alerts published in the past 30 days."

### Cross-Tool Agentic Workflow (Both Structured and Unstructured)

12. "NVDA has been rallying.  What do our quant signals say, what is our current position, and what have analysts written recently?"
13. "Which stocks in our Equity Fund have improving quant scores AND recent analyst upgrades?"
14. "I'm looking for new high-conviction ideas, show me top ranked stocks where analysts also have an Overweight rating."